In [33]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import pandas as pd
import numpy as np

In [34]:
save_path = "../AutogluonModels/TimeSeriesPredictor"
df = pd.read_csv("../data/processed/datas_cleaned.csv")
df["item_id"] = df["region"] + "_" + df["size_rank"].astype(str)
df["timestamp"] = pd.to_datetime((df["year"] - 543).astype(str))
df.head()

,year,region,value,size_rank,tourists_mn,item_id,timestamp
0,2542,Bangkok,"297,061,700",1,3374540,Bangkok_1,1999-01-01
1,2544,Bangkok,"316,149,600",1,3950714,Bangkok_1,2001-01-01
2,2546,Bangkok,"292,475,000",1,5131993,Bangkok_1,2003-01-01
3,2548,Bangkok,"318,711,400",1,4998658,Bangkok_1,2005-01-01
4,2550,Bangkok,"443,951,100",1,4556018,Bangkok_1,2007-01-01


In [35]:
ts_df = TimeSeriesDataFrame.from_data_frame(
    df[["item_id", "timestamp", "value", "tourists_mn"]],
    id_column="item_id",
    timestamp_column="timestamp",
)
ts_df.head()

value  tourists_mn
item_id   timestamp                          
Bangkok_1 1999-01-01 297,061,700      3374540
          2001-01-01 316,149,600      3950714
          2003-01-01 292,475,000      5131993
          2005-01-01 318,711,400      4998658
          2007-01-01 443,951,100      4556018

In [36]:
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="value",
    eval_metric="MASE",
    known_covariates_names=["tourists_mn"],
    path=save_path,
).fit(ts_df, presets="best_quality", time_limit=600)

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to 'd:\Mini Project\Term Project\AutogluonModels\TimeSeriesPredictor'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          20
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       2.59 GB / 15.73 GB (16.5%)
Disk Space Avail:   636.25 GB / 953.85 GB (66.7%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': 'default',
 'known_covariates_names': ['tourists_mn'],
 'num_val_windows': 'auto',
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 'auto',
 

In [37]:
predictor.evaluate(ts_df)

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


{'MASE': -1.6530247746985405}

In [38]:
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")  # จัดรูปแบบตัวเลข
# 1. เตรียมข้อมูลพารามิเตอร์รอบข้างล่วงหน้า (Future Covariates)
future_data = []
for item_id in ts_df.item_ids:
    last_date = ts_df.loc[item_id].index.max()
    future_dates = pd.date_range(start=last_date, periods=3, freq="2YS")[1:]
    region = item_id.split("_")[0]
    for date in future_dates:
        # ยอดนักท่องเที่ยวระดับประเทศช่วงปี 60-62 อยู่ที่ประมาณ 35 ล้านคน
        base = 35.0
        share = {
            "Bangkok": 0.35,
            "Southern": 0.25,
            "Central": 0.15,
            "Northern": 0.15,
            "Northeastern": 0.10,
        }
        # คำนวณยอดของแต่ละภูมิภาค + สุ่ม Noise
        tourists_million = (base * share.get(region, 0.1)) + np.random.uniform(
            -0.5, 0.5
        )
        future_tourists = int(max(0.1, tourists_million) * 1000000)
        future_data.append(
            {"item_id": item_id, "timestamp": date, "tourists_mn": future_tourists}
        )
# 2. แปลงเป็น TimeSeriesDataFrame
future_covariates = TimeSeriesDataFrame.from_data_frame(
    pd.DataFrame(future_data),
    id_column="item_id",
    timestamp_column="timestamp",
)
# 3. สั่งทำนาย โดยใส่ future_covariates เข้าไปด้วย
predictions = predictor.predict(ts_df, known_covariates=future_covariates).reset_index()
predictions

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


,item_id,timestamp,mean,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
0,Bangkok_1,2017-01-01,"2,433,626,390","933,196,028","1,433,868,198","1,771,149,225","2,126,718,910","2,433,626,390","2,901,389,851","3,053,939,007","3,564,777,458","3,900,547,065"
1,Bangkok_1,2019-01-01,"2,480,864,037","952,021,408","1,504,631,706","1,894,371,225","2,249,787,358","2,480,864,037","2,936,559,822","3,133,913,974","3,726,096,516","4,022,056,743"
2,Bangkok_2,2017-01-01,"11,623,507,305","894,561,929","4,203,062,859","6,389,401,374","8,809,016,789","11,623,507,305","14,955,258,578","16,472,090,528","19,439,551,806","22,887,448,598"
3,Bangkok_2,2019-01-01,"12,162,887,152","-2,901,320,030","2,018,358,375","5,332,555,324","8,725,098,282","12,162,887,152","16,148,220,679","18,592,281,203","22,899,475,855","27,827,836,805"
4,Bangkok_3,2017-01-01,"46,609,471,585","28,851,800,840","36,321,052,634","40,310,270,182","44,342,082,587","46,609,471,585","50,437,359,069","53,498,582,263","59,707,486,553","66,160,109,513"
5,Bangkok_3,2019-01-01,"47,199,306,984","29,861,475,731","37,178,080,871","41,053,877,054","44,719,320,210","47,199,306,984","51,256,928,158","54,046,633,906","59,384,908,224","66,422,207,120"
6,Central_1,2017-01-01,"7,296,768,849","467,686,535","2,819,016,099","4,419,536,304","6,100,293,562","7,296,768,849","9,125,967,455","10,162,308,652","12,733,820,707","14,719,798,368"
7,Central_1,2019-01-01,"7,528,855,227","-6,045,464,019","-1,796,563,755","1,109,573,769","4,068,951,797","7,528,855,227","10,925,223,692","13,210,756,546","16,478,881,993","21,076,240,228"
8,Central_2,2017-01-01,"7,669,439,068","3,711,082,041","4,612,426,075","5,268,296,406","6,230,397,173","7,669,439,068","9,140,373,728","9,436,422,561","10,299,397,702","11,557,418,202"
9,Central_2,2019-01-01,"7,720,818,705","3,416,867,241","4,633,628,635","5,659,565,520","6,578,580,345","7,720,818,705","8,930,100,688","9,256,645,492","10,214,006,238","11,514,195,527"


In [39]:
predictions["year_th"] = predictions["timestamp"].dt.year + 543
predictions[["region", "size_rank"]] = predictions["item_id"].str.rsplit(
    "_", n=1, expand=True
)
predictions["size_rank"] = predictions["size_rank"].astype(int)
future_df = pd.DataFrame(future_data)
predictions = pd.merge(predictions, future_df, on=["item_id", "timestamp"], how="left")

In [40]:
actual = df[["item_id", "year", "value", "region", "size_rank", "tourists_mn"]].copy()
actual["year_th"] = actual["year"]
actual["type"] = "actual"

In [41]:
pred_export = predictions[
    ["item_id", "year_th", "region", "size_rank", "mean", "0.1", "0.9", "tourists_mn"]
].copy()
pred_export.columns = [
    "item_id",
    "year_th",
    "region",
    "size_rank",
    "value",
    "lower",
    "upper",
    "tourists_mn",
]
pred_export["type"] = "forecast"

actual_export = actual[
    ["item_id", "year_th", "region", "size_rank", "value", "tourists_mn"]
].copy()
actual_export["lower"] = None
actual_export["upper"] = None
actual_export["type"] = "actual"

combined = pd.concat([actual_export, pred_export], ignore_index=True)
combined.to_csv("../data/predict/predictions.csv", index=False)
print("Saved predictions.csv")

Saved predictions.csv


C:\Users\UNS_CT\AppData\Local\Temp\ipykernel_14952\3417174443.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([actual_export, pred_export], ignore_index=True)
